# 01 — Data Exploration (EDA)

This notebook explores the Fake and Real News Dataset to understand its structure, distribution, and characteristics before preprocessing.

**Objectives:**
- Load and inspect both CSV files
- Analyze class distribution
- Study text lengths
- Detect missing values and duplicates
- Identify potential biases and data leakage risks

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
print("Libraries loaded.")

## 1. Load the Raw Data

We load the two CSV files and assign labels:
- **Fake.csv** → label = 0
- **True.csv** → label = 1

In [ ]:
from src.data_loader import load_raw_data, create_labeled_dataset, validate_dataset

fake_df, true_df = load_raw_data()
print(f"Fake.csv: {fake_df.shape[0]} rows, {fake_df.shape[1]} columns")
print(f"True.csv: {true_df.shape[0]} rows, {true_df.shape[1]} columns")
print(f"Fake.csv columns: {list(fake_df.columns)}")
print(f"True.csv columns: {list(true_df.columns)}")

## 2. Inspect the Data

Let's look at the first few rows and data types.

In [ ]:
print("=== Fake.csv (first 3 rows) ===")
display(fake_df.head(3))
print("\nData types:")
print(fake_df.dtypes)
print("\n=== True.csv (first 3 rows) ===")
display(true_df.head(3))
print("\nData types:")
print(true_df.dtypes)

## 3. Create Labeled and Merged Dataset

In [ ]:
df = create_labeled_dataset(fake_df, true_df)
print(f"Merged dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nLabel mapping: 0 = Fake, 1 = Real")

## 4. Dataset Validation

In [ ]:
validation = validate_dataset(df)
print("Dataset Validation Results:")
for key, value in validation.items():
    print(f"  {key}: {value}")

## 5. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Pct": missing_pct})
print(missing_df)

if "text" in df.columns:
    empty_texts = (df["text"].str.strip() == "").sum()
    print(f"\nEmpty text strings: {empty_texts}")
if "title" in df.columns:
    empty_titles = (df["title"].str.strip() == "").sum()
    print(f"Empty title strings: {empty_titles}")

## 6. Class Distribution

### Visualization 1: Fake vs Real articles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
label_counts = df["label"].value_counts()
label_names = ["Fake (0)", "Real (1)"]
colors = ["#EF5350", "#66BB6A"]

axes[0].bar(label_names, label_counts.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Class Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of Articles")
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 100, str(v), ha="center", fontweight="bold")

axes[1].pie(label_counts.values, labels=label_names, colors=colors,
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Class Proportions", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

fake_pct = label_counts[0] / len(df) * 100
real_pct = label_counts[1] / len(df) * 100
print(f"Fake: {label_counts[0]} ({fake_pct:.1f}%)")
print(f"Real: {label_counts[1]} ({real_pct:.1f}%)")

**Interpretation:** Examine whether the classes are balanced. A roughly balanced dataset means accuracy is a reasonable metric, but precision/recall should still be examined.

## 7. Text Length Analysis

### Visualization 2: Text length distribution by class

In [ ]:
df["text_length"] = df["text"].str.len()
df["title_length"] = df["title"].str.len()
df["word_count"] = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, color, name in [(0, "#EF5350", "Fake"), (1, "#66BB6A", "Real")]:
    subset = df[df["label"] == label]
    axes[0].hist(subset["text_length"], bins=50, alpha=0.6, color=color, label=name)
    axes[1].hist(subset["word_count"], bins=50, alpha=0.6, color=color, label=name)

axes[0].set_title("Text Length Distribution by Class", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].set_title("Word Count Distribution by Class", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/text_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Text length statistics by class:")
print(df.groupby("label")["text_length"].describe().round(0))
print("\nWord count statistics by class:")
print(df.groupby("label")["word_count"].describe().round(0))

**Interpretation:** Examine whether fake and real articles have different length distributions.

### Visualization 3: Title length comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df.boxplot(column="title_length", by="label", ax=ax,
           boxprops=dict(linewidth=1.5),
           medianprops=dict(color="red", linewidth=2))
ax.set_title("Title Length Distribution by Class", fontsize=14, fontweight="bold")
ax.set_xlabel("Class (0=Fake, 1=Real)")
ax.set_ylabel("Title Length (characters)")
plt.suptitle("")
plt.tight_layout()
plt.savefig("../reports/figures/title_length_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Subject Analysis

In [ ]:
if "subject" in df.columns:
    print("Subjects:")
    print(df["subject"].value_counts())
    fig, ax = plt.subplots(figsize=(10, 6))
    sc = df.groupby(["subject", "label"]).size().unstack(fill_value=0)
    sc.columns = ["Fake", "Real"]
    sc.plot(kind="bar", ax=ax, color=["#EF5350", "#66BB6A"], edgecolor="black")
    ax.set_title("Subject Distribution by Class", fontweight="bold")
    ax.set_ylabel("Articles")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig("../reports/figures/subject_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9. Duplicate Analysis

In [ ]:
dup_all = df.duplicated().sum()
dup_text = df.duplicated(subset=["text"]).sum() if "text" in df.columns else 0
dup_tt = df.duplicated(subset=["title", "text"]).sum()
print(f"Exact row duplicates: {dup_all}")
print(f"Duplicate texts: {dup_text}")
print(f"Duplicate (title+text): {dup_tt}")
if dup_tt > 0:
    dup_mask = df.duplicated(subset=["title", "text"], keep=False)
    cross = df[dup_mask].groupby(["title", "text"])["label"].nunique()
    print(f"Cross-class duplicates: {(cross > 1).sum()}")

## 10. Data Leakage Investigation

Some datasets contain markers (like agency names) that make classification trivially easy.

In [ ]:
if "text" in df.columns:
    real = df[df["label"]==1]["text"]
    reuters_real = real.str.contains(r"\(Reuters\)", case=False, na=False).sum()
    fake = df[df["label"]==0]["text"]
    reuters_fake = fake.str.contains(r"\(Reuters\)", case=False, na=False).sum()
    pct_real = reuters_real / len(real) * 100
    pct_fake = reuters_fake / len(fake) * 100
    print(f"Real with (Reuters): {reuters_real}/{len(real)} ({pct_real:.1f}%)")
    print(f"Fake with (Reuters): {reuters_fake}/{len(fake)} ({pct_fake:.1f}%)")

## 11. Sample Articles

In [ ]:
for label, name in [(0, "FAKE"), (1, "REAL")]:
    print(f"\n{'='*60}")
    print(f"Sample {name} Articles")
    sample = df[df["label"]==label].sample(2, random_state=42)
    for _, row in sample.iterrows():
        print(f"\nTitle: {row['title'][:100]}")
        print(f"Text: {row['text'][:200]}...")
        if "subject" in row.index:
            print(f"Subject: {row['subject']}")

## 12. EDA Conclusions

### Key Findings
1. **Dataset size**: 44,898 total raw articles — 23,481 Fake (Fake.csv) and 21,417 Real (True.csv).
2. **Class balance**: Approximately balanced (52.3% Fake vs. 47.7% Real).
3. **Text & title lengths**: Body lengths are comparable (Fake mean 423 words / 2,547 chars; Real mean 386 words / 2,383 chars), while Fake titles are significantly longer (mean 94.2 chars vs. 64.7 chars for Real).
4. **Missing values**: Zero NaNs across all columns, though 631 Fake rows contain whitespace-only body text.
5. **Duplicates**: 5,793 duplicate `(title, text)` pairs (0 cross-class). Must be removed prior to train/test splitting to avoid leakage.
6. **Source markers**: 99.2% of Real articles (21,247/21,417) contain `(Reuters)` vs. 0.04% of Fake (9/23,481) — creating a strong source shortcut that preprocessing mitigates.
7. **Subject distribution**: Fake and Real subjects have zero overlap (`News`, `politics`, `left-news`, etc. vs. `politicsNews`, `worldnews`), so `subject` must NOT be used as a feature.